<a href="https://colab.research.google.com/github/EsarFatima/MachineLearning-flyrank-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EsarFatima/MachineLearning-flyrank-"
REPO_DIR = "MachineLearning-flyrank-"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Engineered features -- same set the modeling lane (w05) relies on.
df["has_keyword_data"] = df["search_volume"].notna().astype(int)   # missingness itself is informative
df["has_word_count"] = df["word_count"].notna().astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])        # impressions are heavy-tailed

num_features = ["content_age_days", "days_since_last_update", "log_impressions_90d",
                 "avg_position", "ctr", "engagement_rate", "search_volume", "competition",
                 "word_count", "has_keyword_data", "has_word_count"]
cat_features = ["content_type", "main_intent"]

print(f"Feature vector: {len(num_features)} numeric + {len(cat_features)} categorical (one-hot)")
print()
print("Missingness (raw, before fill):")
print(df[num_features].isna().mean().round(3))
print()
print("Categorical value counts:")
for c in cat_features:
    print(c, "->", df[c].nunique(), "levels")


Feature vector: 11 numeric + 2 categorical (one-hot)

Missingness (raw, before fill):
content_age_days          0.000
days_since_last_update    0.000
log_impressions_90d       0.000
avg_position              0.000
ctr                       0.000
engagement_rate           0.000
search_volume             0.082
competition               0.082
word_count                0.257
has_keyword_data          0.000
has_word_count            0.000
dtype: float64

Categorical value counts:
content_type -> 3 levels
main_intent -> 4 levels


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
feature_notes = {
    "content_age_days":       "days since first published. No missing. Available at scoring time (static fact about the page).",
    "days_since_last_update": "days since last content edit. No missing. Available at scoring time.",
    "log_impressions_90d":    "log1p of trailing-90d impressions -- a VOLUME level, not a trend. Available at scoring time, "
                               "but see the leakage hunt below: its window overlaps the label's own 30-day window.",
    "avg_position":           "trailing-90d average SERP position. No missing. Available at scoring time.",
    "ctr":                    "trailing-90d click-through rate. No missing. Available at scoring time.",
    "engagement_rate":        "trailing-90d GA4 engagement rate. No missing. Available at scoring time.",
    "search_volume":          "keyword search volume. 8.2% missing -- rows without matched keyword data "
                               "(mostly `feedly article` content_type, confirmed in data-dictionary.md). Filled with 0 "
                               "+ paired has_keyword_data flag so the model can tell 'no data' from 'genuinely zero'.",
    "competition":            "keyword competition score. Same 8.2% missingness pattern as search_volume, same fix.",
    "word_count":             "content length. 25.7% missing. Filled with 0 + has_word_count flag, same reasoning -- "
                               "silently filling 0 without the flag would tell the model 'this page has no words', which is false.",
    "has_keyword_data":       "1/0 flag -- exists purely to make the search_volume/competition fill honest.",
    "has_word_count":         "1/0 flag -- exists purely to make the word_count fill honest.",
    "content_type":           "categorical (3 levels), one-hot encoded, drop_first. Available at scoring time (static).",
    "main_intent":            "categorical (4 levels), one-hot encoded, drop_first. Available at scoring time (static).",
}
for k, v in feature_notes.items():
    print(f"- {k}:")
    print(f"    {v}")


- content_age_days:
    days since first published. No missing. Available at scoring time (static fact about the page).
- days_since_last_update:
    days since last content edit. No missing. Available at scoring time.
- log_impressions_90d:
    log1p of trailing-90d impressions -- a VOLUME level, not a trend. Available at scoring time, but see the leakage hunt below: its window overlaps the label's own 30-day window.
- avg_position:
    trailing-90d average SERP position. No missing. Available at scoring time.
- ctr:
    trailing-90d click-through rate. No missing. Available at scoring time.
- engagement_rate:
    trailing-90d GA4 engagement rate. No missing. Available at scoring time.
- search_volume:
    keyword search volume. 8.2% missing -- rows without matched keyword data (mostly `feedly article` content_type, confirmed in data-dictionary.md). Filled with 0 + paired has_keyword_data flag so the model can tell 'no data' from 'genuinely zero'.
- competition:
    keyword competitio

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler          # <-- add this import up top
from sklearn.metrics import roc_auc_score

# ... Step 1 stays exactly as-is ...

# --- Step 2: the confession test -- train WITH vs WITHOUT the suspect siblings ---
print()
print("Confession test (per the leakage skill): does AUC jump toward 1.0 with the suspect in?")

def make_X(extra_cols):
    num = num_features + extra_cols
    Xn = df[num].replace([np.inf, -np.inf], np.nan).fillna(0)
    Xc = pd.get_dummies(df[cat_features].fillna("unknown"), drop_first=True)
    return pd.concat([Xn, Xc], axis=1)

y = df["is_declining_label"]; groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, y, groups))

for label, extra in [("WITHOUT suspects (clean)", []),
                      ("WITH impressions_last_30d + impressions_prev_30d", ["impressions_last_30d", "impressions_prev_30d"]),
                      ("WITH trend_pct itself (worst case)", ["trend_pct"])]:
    X = make_X(extra)
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    scaler = StandardScaler()                              # <-- these 2 lines are new
    Xtr_s, Xte_s = scaler.fit_transform(Xtr), scaler.transform(Xte)
    m = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
    m.fit(Xtr_s, ytr)                                       # <-- fit on scaled
    auc = roc_auc_score(yte, m.predict_proba(Xte_s)[:, 1])  # <-- predict on scaled
    print(f"  {label}: AUC = {auc:.3f}")

# --- Step 3 stays exactly as-is ---


Confession test (per the leakage skill): does AUC jump toward 1.0 with the suspect in?
  WITHOUT suspects (clean): AUC = 0.541
  WITH impressions_last_30d + impressions_prev_30d: AUC = 0.854
  WITH trend_pct itself (worst case): AUC = 1.000


`trend_pct` is computed directly from `impressions_last_30d` and `impressions_prev_30d`
(correlation = 1.000 — an exact formula, not a coincidence), and `trend_direction` (and therefore
my label `is_declining_label`) is just `trend_pct` cut at ±20%. That makes
`impressions_last_30d`, `impressions_prev_30d`, `trend_pct`, and `trend_direction` all
**label-derived siblings** — the confession test proves it: adding either the two raw windows or
`trend_pct` itself sends AUC from a believable 0.543 straight to a fake-perfect 1.000. None of
the four are used as features anywhere in this project.

`impressions_90d` is a subtler case. It technically **contains** the label's 30-day window (Type-2,
overlapping windows, per the taxonomy) — every row's 90-day total is ≥ its last-30 + prev-30 totals.
But it's a **level** (how much traffic, total), not a **rate of change** (whether traffic moved),
and the numbers back that distinction up: it has essentially zero correlation with `trend_pct`
(-0.003) while having a real, modest correlation with the binary label (0.177) — the same kind of
relationship "big pages behave differently than small pages" would produce with no leakage at all.
I'm keeping `log_impressions_90d` as a feature, but flagging the technical overlap here rather than
pretending it doesn't exist — a stricter version of this feature would use only the 31–90-day-back
sub-window, at the cost of losing recent volume information. That's a real trade-off, not a solved
problem, so I'm carrying the caveat into Week 6 rather than closing it here.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
excluded = {
    "trend_pct":                    "IS the label's raw material -- literally what is_declining_label is thresholded from.",
    "trend_direction":              "IS the label, categorical form.",
    "impressions_last_30d":         "label-derived sibling -- confession test: AUC 0.543 -> 1.000 when included.",
    "impressions_prev_30d":         "label-derived sibling -- same confession test, same reason.",
    "clicks_last_30d / clicks_prev_30d":     "not the exact formula (corr 0.09 with trend_pct) but same 30d/prev-30d "
                                              "window shape as the label -- excluded on the same overlapping-window "
                                              "principle rather than proven necessary; the safer call given how cheap it is.",
    "sessions_last_30d / sessions_prev_30d": "same reasoning as clicks_last_30d/clicks_prev_30d.",
    "provider_used / model_used":   "describe which AI tool generated the content -- a workflow/production detail, not "
                                     "something knowable as a *search performance* signal, and not what this lane is "
                                     "studying. Excluded to keep the feature set about search behavior, not authorship.",
    "content_id / client_id":       "identifiers, not signals -- client_id is used only as the GROUP key for the split, "
                                     "never as a feature (that would leak client identity into the score).",
}
print("Excluded fields:")
for k, v in excluded.items():
    print(f"- {k}: {v}")

print()
print("Final approved feature set (11 numeric + 2 categorical, unchanged from the confession test's 'WITHOUT suspects' run):")
print(" ", num_features + cat_features)

Excluded fields:
- trend_pct: IS the label's raw material -- literally what is_declining_label is thresholded from.
- trend_direction: IS the label, categorical form.
- impressions_last_30d: label-derived sibling -- confession test: AUC 0.543 -> 1.000 when included.
- impressions_prev_30d: label-derived sibling -- same confession test, same reason.
- clicks_last_30d / clicks_prev_30d: not the exact formula (corr 0.09 with trend_pct) but same 30d/prev-30d window shape as the label -- excluded on the same overlapping-window principle rather than proven necessary; the safer call given how cheap it is.
- sessions_last_30d / sessions_prev_30d: same reasoning as clicks_last_30d/clicks_prev_30d.
- provider_used / model_used: describe which AI tool generated the content -- a workflow/production detail, not something knowable as a *search performance* signal, and not what this lane is studying. Excluded to keep the feature set about search behavior, not authorship.
- content_id / client_id: ide

Excluded fields:
- trend_pct: IS the label's raw material -- literally what is_declining_label is thresholded from.
- trend_direction: IS the label, categorical form.
- impressions_last_30d: label-derived sibling -- confession test: AUC 0.543 -> 1.000 when included.
- impressions_prev_30d: label-derived sibling -- same confession test, same reason.
- clicks_last_30d / clicks_prev_30d: not the exact formula (corr 0.09 with trend_pct) but same 30d/prev-30d window shape as the label -- excluded on the same overlapping-window principle rather than proven necessary; the safer call given how cheap it is.
- sessions_last_30d / sessions_prev_30d: same reasoning as clicks_last_30d/clicks_prev_30d.
- provider_used / model_used: describe which AI tool generated the content -- a workflow/production detail, not something knowable as a *search performance* signal, and not what this lane is studying. Excluded to keep the feature set about search behavior, not authorship.
- content_id / client_id: identifiers, not signals -- client_id is used only as the GROUP key for the split, never as a feature (that would leak client identity into the score).

Final approved feature set (11 numeric + 2 categorical, unchanged from the confession test's 'WITHOUT suspects' run):
  ['content_age_days', 'days_since_last_update', 'log_impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'search_volume', 'competition', 'word_count', 'has_keyword_data', 'has_word_count', 'content_type', 'main_intent']
  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.